# BirdCLEF+ 2026 — Google Perch Baseline

Google Perch (`bird-vocalization-classifier v4`) zero-shot baseline.

**Strategy**:
- Load Perch TF SavedModel from Kaggle dataset
- Map 9736 Perch eBird labels → 234 competition species (birds only)
- Sliding window inference (5s windows) on test soundscapes
- Non-Aves species (insects, amphibians, etc.) → uniform score

**Inputs needed**:
- `/kaggle/input/birdclef-2026/` — competition data
- `/kaggle/input/birdclef2026-perch/model/` — Perch TF SavedModel
- `/kaggle/input/birdclef2026-perch/label.csv` — Perch label list

In [ ]:
import os
import glob
import math
import subprocess
import warnings
import numpy as np
import pandas as pd
import soundfile as sf
from pathlib import Path
from tqdm.auto import tqdm

import tensorflow as tf
warnings.filterwarnings('ignore')

# ── CPU threading: use all available cores ─────────────────────────────────────
_ncpu = os.cpu_count() or 4
tf.config.threading.set_intra_op_parallelism_threads(_ncpu)
tf.config.threading.set_inter_op_parallelism_threads(_ncpu)
tf.config.set_visible_devices([], 'GPU')   # CPU-only notebook
print(f'TF {tf.__version__}  |  threads={_ncpu}')

# ── Debug: show ALL mounted datasets ──────────────────────────────────────────
print('=== /kaggle/input/ top-level ===')
for name in sorted(os.listdir('/kaggle/input')):
    p = Path('/kaggle/input') / name
    try:
        children = list(p.iterdir())
        print(f'  {name}/  ({len(children)} items)')
        for c in sorted(children)[:10]:
            print(f'    {c.name}/')
    except Exception as e:
        print(f'  {name}/  ERROR: {e}')

# ── Paths ──────────────────────────────────────────────────────────────────────
OUTPUT      = Path('/kaggle/working')

SAMPLE_RATE = 32_000
WIN_SAMPLES = 5 * SAMPLE_RATE
WIN_STRIDE  = 5 * SAMPLE_RATE
BATCH_SIZE  = 16   # larger batches → better CPU utilisation

# Auto-detect Perch model path (search all of /kaggle/input recursively)
_pb_hits = subprocess.run(['find', '/kaggle/input', '-name', 'saved_model.pb'], capture_output=True, text=True).stdout.strip().split('\n')
_pb_hits = [x for x in _pb_hits if x]
print(f'\nsaved_model.pb hits: {_pb_hits}')
assert _pb_hits, 'saved_model.pb not found anywhere under /kaggle/input/'
PERCH_MODEL = Path(_pb_hits[0]).parent
print(f'Perch model path: {PERCH_MODEL}')

# Auto-detect label.csv
_label_hits = subprocess.run(['find', '/kaggle/input', '-name', 'label.csv'], capture_output=True, text=True).stdout.strip().split('\n')
_label_hits = [x for x in _label_hits if x]
print(f'label.csv hits: {_label_hits}')
assert _label_hits, 'label.csv not found'
PERCH_LABELS = Path(_label_hits[0])
print(f'Perch labels: {PERCH_LABELS}')


In [ ]:
# ── Load Perch TFLite (pre-built offline; avoids runtime conversion overhead) ──
#
# Perch is a JAX→TF converted model and requires the TFLite Flex delegate
# (SELECT_TF_OPS) for ops like Transpose/StridedSlice/EnsureShape.
#
# We search for a pre-built .tflite under /kaggle/input first.
# If not found we convert at runtime (~3-5 min overhead, still worth it).
#
# Batching: we use resize_tensor_input + set_tensor + invoke, NOT
# get_signature_runner.  The latter silently runs at batch=1 for dynamically
# shaped TFLite models — the actual root cause of previous timeouts.

import time as _time

def _build_tflite(saved_model_dir: str) -> bytes:
    print('  Converting SavedModel → TFLite (Flex delegate) …', flush=True)
    conv = tf.lite.TFLiteConverter.from_saved_model(saved_model_dir)
    conv.optimizations = [tf.lite.Optimize.DEFAULT]
    conv.target_spec.supported_ops = [
        tf.lite.OpsSet.TFLITE_BUILTINS,
        tf.lite.OpsSet.SELECT_TF_OPS,
    ]
    conv._experimental_lower_tensor_list_ops = False
    return conv.convert()

# ── Locate pre-built .tflite or build now ─────────────────────────────────────
_tflite_hits = subprocess.run(
    ['find', '/kaggle/input', '-name', '*.tflite'],
    capture_output=True, text=True
).stdout.strip().split('\n')
_tflite_hits = [x for x in _tflite_hits if x]
print(f'.tflite hits: {_tflite_hits}')

if _tflite_hits:
    _tflite_path  = _tflite_hits[0]
    print(f'Loading pre-built TFLite: {_tflite_path}')
    _interp = tf.lite.Interpreter(model_path=_tflite_path, num_threads=_ncpu)
else:
    print('No pre-built .tflite — converting from SavedModel …')
    _tflite_bytes = _build_tflite(str(PERCH_MODEL))
    _interp = tf.lite.Interpreter(model_content=_tflite_bytes, num_threads=_ncpu)

# ── Probe I/O tensors ─────────────────────────────────────────────────────────
_inp_details  = _interp.get_input_details()
_out_details  = _interp.get_output_details()
_inp_idx      = _inp_details[0]['index']
print(f'Inputs:  {[(d["name"], d["shape"]) for d in _inp_details]}')
print(f'Outputs: {[(d["name"], d["shape"]) for d in _out_details]}')

# Warmup with batch=2; find logits output (largest tensor)
_interp.resize_tensor_input(_inp_idx, [2, WIN_SAMPLES])
_interp.allocate_tensors()
_interp.set_tensor(_inp_idx, np.zeros((2, WIN_SAMPLES), dtype=np.float32))
_interp.invoke()
_logits_out   = max(_out_details, key=lambda d: int(np.prod(d['shape'])))
_logits_idx   = _logits_out['index']
N_PERCH       = int(_interp.get_tensor(_logits_idx).shape[-1])
_current_B    = 2

# Benchmark one batch so we can estimate total time
_t0 = _time.time()
_interp.invoke()
_t1 = _time.time()
_batch_sec = _t1 - _t0
print(f'TFLite ready — {N_PERCH} classes | threads={_ncpu} | '
      f'warmup batch=2 → {_batch_sec*1000:.0f} ms', flush=True)

NOTEBOOK_START = _time.time()   # wall-clock reference for timeout guard


In [ ]:
# ── Build species → Perch index mapping ────────────────────────────────────────

# Auto-detect competition data path (search up to 2 levels deep)
COMP_DATA = None
for root in [Path('/kaggle/input'), Path('/kaggle/input/competitions'), Path('/kaggle/input/datasets')]:
    if not root.exists():
        continue
    for p in sorted(root.iterdir()):
        if (p / 'sample_submission.csv').exists():
            COMP_DATA = p
            print(f'Found competition data at: {COMP_DATA}')
            break
    if COMP_DATA:
        break

if COMP_DATA is None:
    import subprocess
    tree = subprocess.run(['find', '/kaggle/input', '-name', 'sample_submission.csv'], capture_output=True, text=True)
    print('find result:', tree.stdout or '(nothing)')
    raise FileNotFoundError('Cannot find sample_submission.csv anywhere under /kaggle/input')

# Competition species (234 labels = column names after row_id)
sub_df     = pd.read_csv(COMP_DATA / 'sample_submission.csv', nrows=0)
COMP_SPECIES = list(sub_df.columns[1:])

# Taxonomy to know which species are Aves (birds) vs invertebrates
tax_df = pd.read_csv(COMP_DATA / 'taxonomy.csv')
is_aves = dict(zip(tax_df['primary_label'], tax_df['class_name'] == 'Aves'))

# Perch eBird label list
perch_labels = pd.read_csv(PERCH_LABELS, skiprows=1, header=None, names=['ebird_code'])
perch_idx    = {code: i for i, code in enumerate(perch_labels['ebird_code'])}

# Map each competition species to its Perch index (-1 = not in Perch)
comp_to_perch = []
n_mapped = 0
for sp in COMP_SPECIES:
    if is_aves.get(sp, False) and sp in perch_idx:
        comp_to_perch.append(perch_idx[sp])
        n_mapped += 1
    else:
        comp_to_perch.append(-1)   # non-Aves or unknown bird → use uniform score

comp_to_perch = np.array(comp_to_perch, dtype=np.int32)

print(f'Competition species: {len(COMP_SPECIES)}')
print(f'Mapped to Perch:     {n_mapped} (of {(np.array([is_aves.get(s,False) for s in COMP_SPECIES])).sum()} Aves)')
print(f'Unmapped (→ 0):      {len(COMP_SPECIES) - n_mapped}')


In [ ]:
# ── Inference helpers ──────────────────────────────────────────────────────────

# Vectorised species mapping (precomputed once)
_valid_mask = comp_to_perch >= 0
_safe_perch = np.where(_valid_mask, comp_to_perch, 0).astype(np.int32)


def _run_batch(windows_np: np.ndarray) -> np.ndarray:
    """Proper batched TFLite invoke; returns (B, 234) sigmoid float32."""
    global _current_B
    B = len(windows_np)
    # Re-allocate only when batch size changes (resize is cheap but allocate is not)
    if B != _current_B:
        _interp.resize_tensor_input(_inp_idx, [B, WIN_SAMPLES])
        _interp.allocate_tensors()
        _current_B = B
    _interp.set_tensor(_inp_idx, windows_np.astype(np.float32))
    _interp.invoke()
    logits = _interp.get_tensor(_logits_idx).copy().astype(np.float32)  # (B, N_PERCH)

    # sigmoid + species mapping — pure numpy, no TF overhead
    probs_perch = 1.0 / (1.0 + np.exp(-np.clip(logits, -88, 88)))
    comp        = probs_perch[:, _safe_perch]     # (B, 234)
    comp[:, ~_valid_mask] = 0.0
    return comp


def load_audio(path: Path) -> np.ndarray:
    data, sr = sf.read(str(path), dtype='float32', always_2d=True)
    data = data.mean(axis=1) if data.shape[1] > 1 else data[:, 0]
    if sr != SAMPLE_RATE:
        raise ValueError(f'Expected 32kHz, got {sr}Hz for {path}')
    mx = np.abs(data).max()
    return data / mx if mx > 0 else data


def infer_windows(waveform: np.ndarray) -> np.ndarray:
    """Returns (n_windows, 234) float32."""
    n_win = math.ceil(len(waveform) / WIN_SAMPLES)
    pad   = n_win * WIN_SAMPLES - len(waveform)
    if pad > 0:
        waveform = np.concatenate([waveform, np.zeros(pad, dtype=np.float32)])
    windows = waveform.reshape(n_win, WIN_SAMPLES)
    return np.concatenate(
        [_run_batch(windows[i: i + BATCH_SIZE]) for i in range(0, n_win, BATCH_SIZE)],
        axis=0
    )


def row_ids_for(stem: str, n_win: int) -> list:
    return [f'{stem}_{(i + 1) * 5}' for i in range(n_win)]


In [ ]:
# ── Run inference on all test soundscapes ─────────────────────────────────────
BUDGET_SECS   = 85 * 60
UNIFORM_SCORE = 1.0 / len(COMP_SPECIES)

test_files = sorted((COMP_DATA / 'test_soundscapes').glob('*.ogg'))
print(f'Test soundscapes: {len(test_files)}')

all_ids   = []
all_probs = []

if not test_files:
    # No test audio present (public run / dry-run) — build uniform submission
    # directly from sample_submission row_ids so the notebook still completes.
    print('No test files found — generating uniform-score placeholder submission.')
    _sample = pd.read_csv(COMP_DATA / 'sample_submission.csv')
    for rid in _sample['row_id']:
        all_ids.append(rid)
        all_probs.append(np.full((1, len(COMP_SPECIES)), UNIFORM_SCORE, dtype=np.float32))
else:
    for fpath in tqdm(test_files, desc='Inference'):
        elapsed = _time.time() - NOTEBOOK_START
        if elapsed > BUDGET_SECS:
            print(f'\n⚠ Time budget reached ({elapsed/60:.1f} min) — '
                  f'filling remaining {len(test_files) - len(all_probs)} files with uniform scores')
            for fp in test_files[len(all_probs):]:
                n_win = 12
                all_ids.extend(row_ids_for(fp.stem, n_win))
                all_probs.append(np.full((n_win, len(COMP_SPECIES)), UNIFORM_SCORE, dtype=np.float32))
            break

        try:
            waveform = load_audio(fpath)
            probs    = infer_windows(waveform)
            all_ids.extend(row_ids_for(fpath.stem, len(probs)))
            all_probs.append(probs)
        except Exception as e:
            print(f'  [warn] {fpath.name}: {e}')
            n_win = 12
            all_ids.extend(row_ids_for(fpath.stem, n_win))
            all_probs.append(np.full((n_win, len(COMP_SPECIES)), UNIFORM_SCORE, dtype=np.float32))

probs_matrix = np.concatenate(all_probs, axis=0)
print(f'Total windows: {len(probs_matrix)}  |  elapsed: {(_time.time()-NOTEBOOK_START)/60:.1f} min')


In [ ]:
# ── Build & validate submission ────────────────────────────────────────────────
# Build directly from the pre-allocated numpy matrix — no list-of-lists overhead
sub = pd.DataFrame(probs_matrix, columns=COMP_SPECIES)
sub.insert(0, 'row_id', all_ids)

# Align to sample_submission row order and fill any missing rows
sample_sub = pd.read_csv(COMP_DATA / 'sample_submission.csv')
sub = sample_sub[['row_id']].merge(sub, on='row_id', how='left')

uniform = 1.0 / len(COMP_SPECIES)
sub[COMP_SPECIES] = sub[COMP_SPECIES].fillna(uniform)

print(f'Submission shape: {sub.shape}')
print(f'Missing values:   {sub.isnull().sum().sum()}')
print(sub.head(3))

out_path = OUTPUT / 'submission.csv'
sub.to_csv(out_path, index=False)
print(f'\nSaved → {out_path}')
